In [1]:
import csv
import pandas as pd
import country_converter as coco
import warnings
warnings.filterwarnings('ignore')  # suppress country_converter 'not found' warnings

In [3]:
# ============================================================
# Cell 2 — Load UNCTAD classification
# ============================================================
# Parent_Code 1500 = Developed  → Global North
# Parent_Code 1400 = Developing → Global South
#
# The file has a double-encoding quirk: each row is stored as one
# quoted string containing all 4 CSV fields. We double-parse to fix it.

records = []
with open('DevelopingVSDevelopedCountries.csv', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    for outer_row in reader:
        inner_row = next(csv.reader([outer_row[0]]))
        if len(inner_row) == 4:
            records.append(inner_row)

df_class_raw = pd.DataFrame(records, columns=header)
df_class_raw['Parent_Code'] = df_class_raw['Parent_Code'].astype(str).str.strip()

developed  = df_class_raw[df_class_raw['Parent_Code'] == '1500'][['Child_Label']].copy()
developing = df_class_raw[df_class_raw['Parent_Code'] == '1400'][['Child_Label']].copy()
developed['global_north_or_south']  = 'Global North'
developing['global_north_or_south'] = 'Global South'

df_class = pd.concat([developed, developing], ignore_index=True)
df_class.columns = ['country', 'global_north_or_south']
df_class['country_key'] = df_class['country'].str.lower().str.strip()

print(f'Classification loaded: {len(df_class)} countries')
print(df_class['global_north_or_south'].value_counts().to_string())

Classification loaded: 264 countries
global_north_or_south
Global South    190
Global North     74


In [5]:
# ============================================================
# Cell 3 — Load trade dataset
# ============================================================
# The CSV has 48 fields per row but only 47 header columns.
# We pad the header with 'extra' to handle this.
# trade_value = cifvalue + fobvalue (nulls treated as 0)
# ISO2 codes are generated via country_converter.

rows = []
with open('TradeValueofCountries.csv', encoding='latin-1') as f:
    reader = csv.reader(f)
    header = next(reader)
    header.append('extra')
    for row in reader:
        if len(row) == 48:
            rows.append(row)
        elif len(row) == 47:
            rows.append(row + [''])

df_trade_raw = pd.DataFrame(rows, columns=header)
df_trade_raw['refYear']  = pd.to_numeric(df_trade_raw['refYear'],  errors='coerce')
df_trade_raw['cifvalue'] = pd.to_numeric(df_trade_raw['cifvalue'], errors='coerce')
df_trade_raw['fobvalue'] = pd.to_numeric(df_trade_raw['fobvalue'], errors='coerce')
df_trade_raw = df_trade_raw[df_trade_raw['refYear'] == 2015].copy()
df_trade_raw['trade_value'] = (
    df_trade_raw['cifvalue'].fillna(0) + df_trade_raw['fobvalue'].fillna(0)
)

df_trade = df_trade_raw[['reporterDesc', 'trade_value']].dropna(subset=['reporterDesc']).copy()
df_trade.columns = ['country', 'trade_value']

cc = coco.CountryConverter()
df_trade['iso2'] = cc.convert(names=df_trade['country'], to='ISO2')
df_trade['country_key'] = df_trade['country'].str.lower().str.strip()

print(f'Trade data loaded: {len(df_trade)} countries (2015)')
print(df_trade.head(5).to_string(index=False))

Other Asia, nes not found in regex


Trade data loaded: 175 countries (2015)
    country  trade_value iso2 country_key
Afghanistan 5.714050e+08   AF afghanistan
    Albania 1.929657e+09   AL     albania
    Algeria 3.479595e+10   DZ     algeria
    Andorra 8.950194e+07   AD     andorra
     Angola 3.392494e+10   AO      angola


In [13]:
# ============================================================
# Cell 4 — Load HDI dataset
# ============================================================
# Semicolon-separated, European decimal format (comma as decimal point).
# Column headers are on row 5 (header=4). The 2015 HDI column is named '2015'.
# Rows without a numeric HDI rank are group headers or footnotes — skipped.

df_hdi_raw = pd.read_csv(
    'HDIofCountries.csv',
    sep=';',
    encoding='latin-1',
    header=4,
    decimal=','
)
df_hdi_raw = df_hdi_raw[['HDI rank', 'Country', '2015']].copy()
df_hdi_raw.columns = ['hdi_rank', 'country', 'HDI']

df_hdi = df_hdi_raw[pd.to_numeric(df_hdi_raw['hdi_rank'], errors='coerce').notna()].copy()
df_hdi = df_hdi.dropna(subset=['country', 'HDI'])
df_hdi['country_key'] = df_hdi['country'].str.lower().str.strip()
df_hdi = df_hdi[['country_key', 'HDI']]

# Convert HDI to float — the European decimal format (comma) can cause it
# to be stored as a string like '0,956' even after decimal=',' is set.
df_hdi['HDI'] = df_hdi['HDI'].astype(str).str.replace(',', '.', regex=False)
df_hdi['HDI'] = pd.to_numeric(df_hdi['HDI'], errors='coerce')

print(f'HDI data loaded: {len(df_hdi)} countries (2015)')

HDI data loaded: 193 countries (2015)


In [15]:
# ============================================================
# Cell 5 — Alias maps
# ============================================================
# Each dataset uses different naming conventions for the same countries.
# These maps translate between them so the merges work correctly.
#
# Diagnosis of every missing country:
#   FIXABLE WITH ALIAS (exist in all 3 datasets under different names):
#     bolivia, bosnia and herzegovina, brunei, cape verde,
#     central african republic, dominican republic, dr congo, eswatini,
#     hong kong, iran, ivory coast, laos, moldova, netherlands,
#     palestine, republic of the congo, russia, south korea,
#     tanzania, turkey, united states, venezuela, vietnam
#
#   NOT IN TRADE DATA AT ALL (cannot be recovered — no 2015 Comtrade record):
#     bhutan, chad, cuba, djibouti, dominica, equatorial guinea, eritrea,
#     liberia, libya, liechtenstein, mali, marshall islands, micronesia,
#     monaco, nauru, north korea, papua new guinea, san marino,
#     solomon islands, somalia, south sudan, syria, timor-leste, tonga,
#     turkmenistan, tuvalu, uzbekistan, vanuatu, vatican city

# trade_name_lower → unctad_key_lower
trade_to_unctad = {
    'usa'                                  : 'united states',
    'russian federation'                   : 'russian federation',
    'türkiye'                              : 'turkiye',
    'rep. of korea'                        : 'republic of korea',
    'viet nam'                             : 'viet nam',
    'united rep. of tanzania'              : 'united republic of tanzania',
    'bolivia (plurinational state of)'     : 'bolivia (plurinational state of)',
    'dem. rep. of the congo'               : 'dem. rep. of the congo',
    'congo'                                : 'congo',
    "côte d'ivoire"                        : "cote d'ivoire",
    'china, hong kong sar'                 : 'china, hong kong sar',
    'rep. of moldova'                      : 'republic of moldova',
    "lao people's dem. rep."               : "lao people's dem. rep.",
    'brunei darussalam'                    : 'brunei darussalam',
    'cabo verde'                           : 'cabo verde',
    'iran'                                 : 'iran (islamic republic of)',
    'state of palestine'                   : 'state of palestine',
    'bosnia herzegovina'                   : 'bosnia and herzegovina',
    'central african rep.'                 : 'central african republic',
    'dominican rep.'                       : 'dominican republic',
    'netherlands'                          : 'netherlands (kingdom of the)',
    'eswatini'                             : 'eswatini',
}

# trade_name_lower → hdi_key_lower
trade_to_hdi = {
    'usa'                                  : 'united states',
    'russian federation'                   : 'russian federation',
    'türkiye'                              : 'türkiye',
    'rep. of korea'                        : 'korea (republic of)',
    'viet nam'                             : 'viet nam',
    'united rep. of tanzania'              : 'tanzania (united republic of)',
    'bolivia (plurinational state of)'     : 'bolivia (plurinational state of)',
    'dem. rep. of the congo'               : 'congo (democratic republic of the)',
    'congo'                                : 'congo',
    "côte d'ivoire"                        : "côte d'ivoire",
    'china, hong kong sar'                 : 'hong kong, china (sar)',
    'rep. of moldova'                      : 'moldova (republic of)',
    "lao people's dem. rep."               : "lao people's democratic republic",
    'brunei darussalam'                    : 'brunei darussalam',
    'cabo verde'                           : 'cabo verde',
    'iran'                                 : 'iran (islamic republic of)',
    'state of palestine'                   : 'palestine, state of',
    'bosnia herzegovina'                   : 'bosnia and herzegovina',
    'central african rep.'                 : 'central african republic',
    'dominican rep.'                       : 'dominican republic',
    'eswatini'                             : 'eswatini (kingdom of)',
}

# Friendly display name overrides (trade name → clean display name)
display_name_override = {
    'usa'                              : 'United States',
    'russian federation'               : 'Russia',
    'türkiye'                          : 'Turkey',
    'rep. of korea'                    : 'South Korea',
    'viet nam'                         : 'Vietnam',
    'united rep. of tanzania'          : 'Tanzania',
    'bolivia (plurinational state of)' : 'Bolivia',
    'dem. rep. of the congo'           : 'DR Congo',
    'congo'                            : 'Republic of the Congo',
    "côte d'ivoire"                    : "Ivory Coast",
    'china, hong kong sar'             : 'Hong Kong',
    'rep. of moldova'                  : 'Moldova',
    "lao people's dem. rep."           : 'Laos',
    'brunei darussalam'                : 'Brunei',
    'cabo verde'                       : 'Cape Verde',
    'state of palestine'               : 'Palestine',
    'bosnia herzegovina'               : 'Bosnia and Herzegovina',
    'central african rep.'             : 'Central African Republic',
    'dominican rep.'                   : 'Dominican Republic',
}

print('Alias maps defined.')

Alias maps defined.


In [17]:
# ============================================================
# Cell 6 — Resolve and merge
# ============================================================
# Instead of relying on raw inner joins (which drop countries whenever
# names differ even slightly), we resolve each trade country name to
# the equivalent key in UNCTAD and HDI via the alias maps, then look up
# values directly. This recovers all 21 fixable countries.

unctad_lookup = df_class.set_index('country_key')['global_north_or_south'].to_dict()
hdi_lookup    = df_hdi.set_index('country_key')['HDI'].to_dict()

results = []
skipped = []

for _, row in df_trade.iterrows():
    trade_key  = row['country_key']
    iso2       = row['iso2']
    trade_val  = row['trade_value']

    # Resolve to UNCTAD and HDI keys via alias maps (fall back to same name)
    unctad_key = trade_to_unctad.get(trade_key, trade_key)
    hdi_key    = trade_to_hdi.get(trade_key, trade_key)

    classification = unctad_lookup.get(unctad_key)
    hdi_val        = hdi_lookup.get(hdi_key)

    if classification is None or hdi_val is None:
        skipped.append({
            'country'      : row['country'],
            'unctad_found' : classification is not None,
            'hdi_found'    : hdi_val is not None,
        })
        continue

    # Use clean display name if one is defined, otherwise use trade name as-is
    display_name = display_name_override.get(trade_key, row['country'])

    results.append({
        'iso2'                 : iso2,
        'country'              : display_name,
        'global_north_or_south': classification,
        'trade_value'          : trade_val,
        'HDI'                  : hdi_val,
    })

df_final = pd.DataFrame(results).sort_values('country').reset_index(drop=True)

print(f'Final dataset: {len(df_final)} countries')
print(f'Skipped (no UNCTAD or HDI match): {len(skipped)}')
if skipped:
    print('Skipped countries:')
    print(pd.DataFrame(skipped).to_string(index=False))
print()
print(df_final.head(10).to_string(index=False))

Final dataset: 166 countries
Skipped (no UNCTAD or HDI match): 9
Skipped countries:
         country  unctad_found  hdi_found
         Bermuda          True      False
    Solomon Isds         False      False
French Polynesia          True      False
       Greenland          True      False
China, Macao SAR          True      False
 Other Asia, nes         False      False
      Montserrat          True      False
           Aruba          True      False
   New Caledonia          True      False

iso2             country global_north_or_south  trade_value   HDI
  AF         Afghanistan          Global South 5.714050e+08 0.496
  AL             Albania          Global North 1.929657e+09 0.797
  DZ             Algeria          Global South 3.479595e+10 0.737
  AD             Andorra          Global North 8.950194e+07 0.869
  AO              Angola          Global South 3.392494e+10 0.603
  AG Antigua and Barbuda          Global South 2.604554e+07 0.839
  AR           Argentina         

In [ ]:
# ============================================================
# Cell 7 — Save to CSV
# ============================================================

df_final.to_csv('StructuralFactorsOfCountries.csv', index=False, encoding='utf-8-sig')

print(f'Saved: StructuralFactorsOfCountries.csv')
print(f'  Rows    : {len(df_final)}')
print(f'  Columns : {df_final.columns.tolist()}')